# Step 1 — OCR + GPT Extraction → Azure DeID (Surrogate), outputs saved next to the image

The first building block of the de-identification pipeline. **No image is changed yet.**

| Step | What happens |
| --- | --- |
| 1 | **Textract `DetectDocumentText`** reads the image → every word with its text, pixel box, tilt, confidence, printed/handwriting |
| 2 | The **OCR text** is rebuilt line by line (`build_word_index`); every word keeps its word id, its position in the text and its pixel box |
| 3 | **GPT-5.4** reads the image → JSON: every filled-in field (`label`, `value`, `category`, `is_phi`) and PHI outside fields (`free_text_phi`) |
| 4 | The **GPT text** is built from GPT's own values, one line per item: `LABEL: value` for fields, `Category: value` for free-text PHI |
| 5 | Both texts go to **Azure DeID `Surrogate`** separately (2 calls); Azure returns each text with every PHI value replaced by a fake |
| 6 | Each GPT item is paired with Azure's value for it (line by line) → changed / unchanged |
| 7 | Everything is **saved in the same folder as the input image** |

**Files written next to `<name>.png`** (`OUTPUT_SUBFOLDER = ""` = the image's own folder):

| File | Contents | Holds real PHI? |
| --- | --- | --- |
| `<name>_ocr_text.txt` | the rebuilt OCR text = what is sent to Azure (call 1) | **yes** |
| `<name>_ocr_words.json` | every OCR word: word id, text, line, position in the text, pixel box, confidence, handwriting, tilt | **yes** |
| `<name>_gpt.json` | GPT-5.4's answer exactly as returned (or the error) | **yes** |
| `<name>_gpt_text.txt` | the GPT text = what is sent to Azure (call 2) | **yes** |
| `<name>_deid_ocr_text.txt` | Azure's Surrogate output for the OCR text | fakes only (PHI Azure missed stays) |
| `<name>_deid_gpt_text.txt` | Azure's Surrogate output for the GPT text | fakes only (PHI Azure missed stays) |
| `<name>_gpt_deid_items.json` | per GPT item: label, original value, Azure's value, changed yes/no, category, is_phi | **yes** |
| `<name>_run_info.json` | counts, Azure DeID calls and bytes, time per step, errors | no |

**Keep the folder protected:** most of these files contain the original PHI.

## 1. Libraries

In [ ]:
import boto3        # AWS SDK for Python — Textract OCR
import json          # JSON parsing for API responses
import os            # File path operations
from collections import Counter  # Block type counting
from icp.authenticator.aws.boto3_services import boto3_clients as aws_client
import ssl           # Certificate handling
import sys           # Platform detection
import tempfile      # Temporary file creation
import urllib.error  # HTTP error handling
import urllib.request  # HTTP requests

import re
import time
from pathlib import Path

print("✓ All libraries imported successfully")

## 2. Azure DeID Configuration (.env)

| `.env` key | Meaning |
| --- | --- |
| `AZURE_DEID_ENDPOINT`, `AZURE_DEID_TENANT_ID` | required: DeID service URL and tenant |
| `AZURE_DEID_SCOPE`, `AZURE_DEID_API_VERSION` | optional (defaults `https://deid.azure.com/.default`, `2024-11-15`) |
| `endpoint`, `deployment`, `subscription_key`, `api_version`, `model_name` | GPT-5.4 (Azure OpenAI), read in section 6 |

In [ ]:
# Azure DeID Service Configuration — loaded from .env (keeps secrets out of the notebook)
from dotenv import load_dotenv

load_dotenv()  # reads the .env file in the project root

ENDPOINT = os.environ["AZURE_DEID_ENDPOINT"]
TENANT_ID = os.environ["AZURE_DEID_TENANT_ID"]
SCOPE = os.environ.get("AZURE_DEID_SCOPE", "https://deid.azure.com/.default")
API_VERSION = os.environ.get("AZURE_DEID_API_VERSION", "2024-11-15")

print(f"Endpoint: {ENDPOINT}")
print(f"Tenant ID: {TENANT_ID}")
print(f"API Version: {API_VERSION}")

## 3. Company Certificates (Netskope)

Builds one certificate file = Python's standard list + the Netskope certificates from the Windows store, so AWS, Azure and OpenAI calls work on the company network.

In [ ]:
import certifi

def export_netskope_ca():
    """Write a CA bundle containing certifi roots plus any Netskope roots from the Windows store."""
    base_bundle = certifi.where()
    bundle_text = ""
    if base_bundle and os.path.exists(base_bundle):
        with open(base_bundle, "r", encoding="ascii", errors="ignore") as f:
            bundle_text = f.read()
    if sys.platform != "win32":
        return base_bundle if bundle_text else None

    pems = []
    for store in ("ROOT", "CA"):
        for der, encoding, _ in ssl.enum_certificates(store):
            if encoding != "x509_asn":
                continue
            pem = ssl.DER_cert_to_PEM_cert(der)
            if b"netskope" in der.lower() or b"goskope" in der.lower():
                pems.append(pem)

    if not pems:
        return base_bundle if bundle_text else None

    path = os.path.join(tempfile.gettempdir(), "NetskopeRootPlusCertifi.pem")
    with open(path, "w", encoding="ascii") as f:
        if bundle_text:
            f.write(bundle_text.rstrip() + "\n")
        f.write("\n".join(pems))
    return path

def configure_ca_bundle(explicit=None):
    bundle = explicit or os.environ.get("REQUESTS_CA_BUNDLE") or os.environ.get("SSL_CERT_FILE") or os.environ.get("AWS_CA_BUNDLE")
    if not bundle:
        bundle = export_netskope_ca()
        if bundle:
            print(f"Using CA bundle at {bundle}")
    if bundle:
        os.environ["REQUESTS_CA_BUNDLE"] = bundle
        os.environ["SSL_CERT_FILE"] = bundle
        os.environ["AWS_CA_BUNDLE"] = bundle
    else:
        print("No custom CA bundle in use (none provided, none found).")
    return bundle

# Force a freshly generated bundle so the notebook does not keep reusing an older temp path.
ca_bundle = configure_ca_bundle(export_netskope_ca())
print(f"✓ CA bundle configured: {ca_bundle if ca_bundle else 'Default'}")

## 4. Azure Sign-in

Opens a browser; sign in with an account that has the **DeID Data Owner** role. The token is kept and renewed automatically.

In [ ]:
from azure.identity import InteractiveBrowserCredential

print("Opening browser for sign-in...")
print("(A browser window will open. Login with your Azure account that has DeID Data Owner role)\n")

credential = InteractiveBrowserCredential(tenant_id=TENANT_ID, timeout=240)

try:
    token = credential.get_token(SCOPE)
    print("✓ Authentication: OK")
    print(f"Token expires in: {token.expires_on}")
except Exception as exc:
    print(f"❌ FAILED: authentication error:\n{exc}")
    raise

## 5. Textract OCR + OCR Text

- `get_ocr(image_bytes)`: Textract `DetectDocumentText` (basic OCR, about $0.0015 per page; image under 10 MB). `@aws_client.refresh_on_expire` renews expired AWS credentials.
- `build_word_index(blocks, img_w, img_h)`: rebuilds the page text line by line (`full_text`) and records for every word its `start`/`end` in that text, word id, line, confidence, handwriting flag, pixel box (`px`) and tilt (`angle`). `full_text[start:end]` is always exactly the word, so anything Azure changes in the text can later be traced to the word and its pixels.

In [ ]:
@aws_client.refresh_on_expire
def get_ocr(image_bytes):
    """Basic OCR: lines + words with boxes, confidence and handwriting flag (Textract DetectDocumentText)."""
    textract = aws_client.get("textract")
    return textract.detect_document_text(Document={'Bytes': image_bytes})


# Build the word index: reconstruct the page text line by line and remember, for
# every character range, which WORD box it came from.
# Added vs. the old version: pixel box, polygon, line number, skew angle and handwriting flag.
import math
from PIL import Image

def build_word_index(blocks, img_w, img_h):
    """Returns (full_text, offset_map). offset_map is in reading order."""
    block_map = {b['Id']: b for b in blocks} 
    line_blocks = [b for b in blocks if b['BlockType'] == 'LINE']

    full_text = ""
    offset_map = []

    for line_idx, line in enumerate(line_blocks):
        word_ids = []
        for rel in line.get('Relationships', []):
            if rel['Type'] == 'CHILD':
                word_ids.extend(rel['Ids'])

        for word_id in word_ids:
            word_block = block_map.get(word_id)
            if not word_block or word_block['BlockType'] != 'WORD':
                continue  # skip SELECTION_ELEMENT or other non-word children

            word_text = word_block['Text']
            bbox = word_block['Geometry']['BoundingBox']
            poly = word_block['Geometry'].get('Polygon') or []

            # Skew of this word in degrees (0 = horizontal), from the top edge of its polygon
            angle = 0.0
            if len(poly) >= 2:
                dx = (poly[1]['X'] - poly[0]['X']) * img_w
                dy = (poly[1]['Y'] - poly[0]['Y']) * img_h
                if abs(dx) + abs(dy) > 0:
                    angle = math.degrees(math.atan2(dy, dx))

            start = len(full_text)
            full_text += word_text
            end = len(full_text)

            offset_map.append({
                "idx": len(offset_map),
                "start": start,
                "end": end,
                "text": word_text,
                "bbox": bbox,
                "confidence": word_block['Confidence'],
                "word_id": word_id,
                "line_idx": line_idx,
                "px": (int(math.floor(bbox['Left'] * img_w)),
                       int(math.floor(bbox['Top'] * img_h)),
                       int(math.ceil((bbox['Left'] + bbox['Width']) * img_w)),
                       int(math.ceil((bbox['Top'] + bbox['Height']) * img_h))),
                "poly": [(pt['X'] * img_w, pt['Y'] * img_h) for pt in poly] if len(poly) >= 3 else None,
                "angle": angle,
                "handwriting": word_block.get('TextType') == 'HANDWRITING',
            })
            full_text += " "   # space between words
        full_text += "\n"      # newline between lines

    return full_text, offset_map



print("✓ OCR functions defined")

## 6. GPT-5.4 Extraction + GPT Text

- `vlm_extract(image_path)`: the image (resized to ≤ `VLM_MAX_WIDTH` px wide) + `VLM_PROMPT` → GPT-5.4 → JSON with `fields` (every filled-in field: label, value, category, is_phi), `free_text_phi` (PHI outside labelled fields: notes, stamps, signatures, headers) and `printed_text`.
- `build_gpt_text(gpt)`: one line per item, using **GPT's own values**: `LABEL: value` for every field (PHI or not, as context), `Category: value` for every free-text PHI value. Returns the text and the list of items (which line is which item), so Azure's answer can be paired back to each item.

In [ ]:
# GPT-5.4 (Azure OpenAI) reads the page image and returns label / value / category for every filled-in field.
# Values are matched back to the Textract words, so the VLM adds CONTEXT, while Textract keeps the POSITIONS.
import base64
import io
import re
from difflib import SequenceMatcher
from pathlib import Path

from openai import AzureOpenAI
from PIL import Image

# ---------------------------------------------------------------- settings
USE_VLM = True
VLM_MAX_WIDTH = 2000          # image is resized to at most this width before sending
VLM_MAX_TOKENS = 16000        # GPT-5.x counts reasoning tokens here too
VLM_MATCH_THRESHOLD = 0.85    # how similar a VLM value must be to OCR words to count as the same text
VLM_CACHE_DIR = None          # e.g. Path(r"C:\...\vlm_cache") to reuse answers on re-runs (contains PHI)

# ---------------------------------------------------------------- Azure OpenAI settings from .env
AZURE_OPENAI_ENDPOINT = os.getenv("endpoint") or os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_DEPLOYMENT = os.getenv("deployment") or os.getenv("AZURE_OPENAI_DEPLOYMENT")
AZURE_OPENAI_API_KEY = os.getenv("subscription_key") or os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_API_VERSION = os.getenv("api_version") or os.getenv("AZURE_OPENAI_API_VERSION") or "2024-10-21"
VLM_MODEL_NAME = os.getenv("model_name") or os.getenv("model") or "gpt-5.4"     # for logging only

_missing = [n for n, v in {"endpoint": AZURE_OPENAI_ENDPOINT, "deployment": AZURE_OPENAI_DEPLOYMENT,
                           "subscription_key": AZURE_OPENAI_API_KEY}.items() if not v]
if USE_VLM and _missing:
    raise ValueError(f"Missing Azure OpenAI settings in .env: {', '.join(_missing)} (or set USE_VLM = False)")

vlm_client = AzureOpenAI(api_version=AZURE_OPENAI_API_VERSION, azure_endpoint=AZURE_OPENAI_ENDPOINT,
                         api_key=AZURE_OPENAI_API_KEY) if USE_VLM else None

VLM_PROMPT = """You are reading a scanned healthcare document (for example a CMS-1500 or UB-04 claim form).
Return JSON only, in this shape:
{"fields": [{"label": "...", "value": "...", "category": "...", "is_phi": true}],
 "free_text_phi": [{"value": "...", "category": "..."}],
 "printed_text": ["..."]}

fields: every FILLED-IN field on the page.
- "label": the printed field name exactly as printed, including its box or form-locator number
  (e.g. "2. PATIENT'S NAME (Last Name, First Name, Middle Initial)", "10 BIRTHDATE", "45 SERV. DATE").
- "value": the typed or handwritten value for that field, copied EXACTLY as printed: same spelling, spacing,
  punctuation and date format. Do not correct, reformat or complete anything.
- One entry per field. For tables (e.g. UB-04 service lines) give one entry per filled cell that holds an
  identifier or a date, with the column header as label.
- If one box holds a full address, give street, city, state and zip as separate entries with the same label.
- "category": one of name, date, age_over_89, phone, fax, email, ssn, medical_record_number, health_plan_id,
  account_number, license_number, id_number, street, city, state, zip, country, organization, hospital,
  profession, url, other.
- "is_phi": true for identifiers of a patient, insured person, guarantor, relative or provider (names, dates,
  addresses, phone numbers, ID / account / policy / member / license / NPI numbers); false for diagnosis or
  procedure codes, charges, units, checkbox marks, yes/no, sex, relationship, and printed form text.

free_text_phi: identifiers that are NOT in a labelled field (notes, stamps, signatures, headers), copied exactly.

printed_text: every PRE-PRINTED piece of text that belongs to the blank form itself, one entry per printed line,
copied exactly: form titles, approval lines, box captions and labels, checkbox captions (e.g. MEDICARE, MEDICAID,
TRICARE, SSN, EIN, YES, NO), column headers, date captions (MM DD YY), instructions, footers, form numbers.
Never include anything that was typed or written into the form.

Do not list empty fields. Do not invent values.
Return {"fields": [], "free_text_phi": [], "printed_text": []} if nothing is filled in."""

# VLM category -> kind used by section 12 / 13
VLM_KIND = {
    "name": "name", "date": "date", "phone": "phone", "fax": "phone", "email": "email",
    "street": "street", "city": "city", "state": "state", "zip": "zip", "country": "country",
    "organization": "org", "hospital": "org", "profession": "profession",
}   # everything else (ssn, *_number, health_plan_id, age_over_89, url, other) -> "id"


def vlm_kind(category, value=""):
    c = str(category or "").strip().lower()
    if c == "age_over_89":
        digits = re.sub(r"\D", "", str(value))
        return "id" if digits and int(digits) > 89 else None
    return VLM_KIND.get(c, "id")


# ---------------------------------------------------------------- calling GPT-5.4
def _image_b64(image_path, max_width=VLM_MAX_WIDTH):
    image = Image.open(image_path).convert("RGB")
    if image.width > max_width:
        scale = max_width / image.width
        image = image.resize((int(image.width * scale), int(image.height * scale)), Image.Resampling.LANCZOS)
    buffer = io.BytesIO()
    image.save(buffer, format="PNG")
    return base64.b64encode(buffer.getvalue()).decode("utf-8")


def _parse_json(text_output):
    cleaned = (text_output or "").strip()
    if cleaned.startswith("```"):
        cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
        cleaned = re.sub(r"\s*```$", "", cleaned)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", cleaned, re.S)
        if not match:
            raise
        return json.loads(match.group(0))


def vlm_extract(image_path):
    """GPT-5.4 answer for one page: {"fields": [...], "free_text_phi": [...]}. Uses the cache if set."""
    cache = Path(VLM_CACHE_DIR) / f"{Path(image_path).stem}_vlm.json" if VLM_CACHE_DIR else None
    if cache and cache.exists():
        return json.loads(cache.read_text(encoding="utf-8"))
    response = vlm_client.chat.completions.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        messages=[
            {"role": "system", "content": "You read healthcare document images and return strict JSON only."},
            {"role": "user", "content": [
                {"type": "text", "text": VLM_PROMPT},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{_image_b64(image_path)}",
                                                    "detail": "high"}},
            ]},
        ],
        max_completion_tokens=VLM_MAX_TOKENS,
    )
    choice = response.choices[0]
    if choice.finish_reason == "length":
        raise RuntimeError(f"VLM output cut off at {VLM_MAX_TOKENS} tokens; raise VLM_MAX_TOKENS")
    result = _parse_json(choice.message.content)
    result.setdefault("fields", []); result.setdefault("free_text_phi", []); result.setdefault("printed_text", [])
    if cache:
        cache.parent.mkdir(parents=True, exist_ok=True)
        cache.write_text(json.dumps(result, indent=1, ensure_ascii=False), encoding="utf-8")
    return result




# ---------------------------------------------------------------- GPT text: one line per GPT item, GPT's own values
CATEGORY_LABELS = {
    "name": "Name", "date": "Date", "age_over_89": "Age", "phone": "Phone", "fax": "Fax", "email": "Email",
    "ssn": "SSN", "medical_record_number": "Medical record number", "health_plan_id": "Health plan ID",
    "account_number": "Account number", "license_number": "License number", "id_number": "ID number",
    "street": "Street address", "city": "City", "state": "State", "zip": "ZIP code", "country": "Country",
    "organization": "Organization", "hospital": "Hospital", "profession": "Profession", "url": "URL",
}

def _one_line(s):
    return re.sub(r"\s+", " ", str(s or "")).strip()

def build_gpt_text(gpt):
    """(text, items): one 'LABEL: value' line per GPT field and one 'Category: value' line per free-text PHI."""
    lines_out, items = [], []
    for f in gpt.get("fields", []) or []:
        value = _one_line(f.get("value"))
        if not value:
            continue
        label = _one_line(f.get("label")) or "Value"
        items.append({"line": len(lines_out), "source": "field", "label": label, "value": value,
                      "category": f.get("category"), "is_phi": f.get("is_phi", True)})
        lines_out.append(f"{label}: {value}")
    for p in gpt.get("free_text_phi", []) or []:
        value = _one_line(p.get("value"))
        if not value:
            continue
        cat = str(p.get("category") or "other").strip().lower()
        label = CATEGORY_LABELS.get(cat, cat.replace("_", " ").title())
        items.append({"line": len(lines_out), "source": "free_text", "label": label, "value": value,
                      "category": cat, "is_phi": True})
        lines_out.append(f"{label}: {value}")
    return ("\n".join(lines_out) + "\n") if lines_out else "", items

print("✓ GPT functions defined")

## 7. Azure DeID (Surrogate)

- `call_azure_deidentification(text, "Surrogate")`: one request; returns Azure's JSON (`outputText` = the text with every PHI value replaced). Every call is counted in `AZURE_USAGE` (calls, bytes sent) for the cost.
- `_split_chunks`: text over `AZURE_MAX_CHARS` (20,000) is cut at line breaks; a normal page is one call per text.
- `deid_surrogate(text)`: runs the chunks and returns Azure's full output text.
- `pair_gpt_items(items, sent, returned)`: for every GPT item, Azure's value on the same line (`changed` = Azure replaced it). If Azure changed the number of lines, the pairing is skipped and reported.

In [ ]:
# Azure DeID REST call (Surrogate). A token is requested for every call (azure-identity caches and renews it);
# every call is counted in AZURE_USAGE because Azure bills per MB of text de-identified.

AZURE_MAX_CHARS = 20000   # text is sent in chunks of at most this many characters
AZURE_USAGE = {"calls": 0, "input_bytes": 0}   # billed data: counted before every DeID call

def call_azure_deidentification(text, operation="Surrogate"):
    """One Azure DeID request (this notebook uses "Surrogate"). Returns the JSON answer."""
    AZURE_USAGE["calls"] += 1
    AZURE_USAGE["input_bytes"] += len(text.encode("utf-8"))   # size of the text actually sent
    token = credential.get_token(SCOPE)
    body = json.dumps({"inputText": text, "operation": operation}).encode()
    request = urllib.request.Request(
        f"{ENDPOINT}/deid?api-version={API_VERSION}",
        data=body,
        method="POST",
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {token.token}",
        },
    )
    try:
        with urllib.request.urlopen(request, timeout=60) as response:
            return json.loads(response.read())
    except urllib.error.HTTPError as exc:
        print(f"❌ Azure DeID error: {exc.code} {exc.reason}")
        print(exc.read().decode(errors="replace")[:500])
        raise


def _split_chunks(text, limit):
    """Split on line boundaries so no entity is cut in half."""
    chunks, pos = [], 0
    while pos < len(text):
        end = min(len(text), pos + limit)
        if end < len(text):
            nl = text.rfind("\n", pos, end)
            if nl > pos:
                end = nl + 1
        chunks.append((pos, text[pos:end]))
        pos = end
    return chunks



def deid_surrogate(text):
    """Azure DeID Surrogate on a whole text (chunked at line breaks). Returns Azure's output text."""
    if not text.strip():
        return ""
    out = []
    for _base, chunk in _split_chunks(text, AZURE_MAX_CHARS):
        if not chunk.strip():
            out.append(chunk)
            continue
        result = call_azure_deidentification(chunk, "Surrogate")
        surrogate = result.get("outputText") or result.get("output_text")
        if surrogate is None:
            raise RuntimeError(f"No outputText in the Azure Surrogate response. Keys returned: {list(result)}")
        out.append(surrogate)
    return "".join(out)


def pair_gpt_items(items, sent, returned):
    """Azure's value for each GPT item, taken from the same line of Azure's output."""
    s_lines, r_lines = sent.split("\n"), returned.split("\n")
    if len(s_lines) != len(r_lines):
        for it in items:
            it.update({"deid_value": None, "changed": None, "note": "Azure changed the number of lines"})
        return False
    for it in items:
        sent_line, ret_line = s_lines[it["line"]], r_lines[it["line"]]
        prefix = it["label"] + ": "
        if ret_line.startswith(prefix):
            new_value = ret_line[len(prefix):].strip()
        elif ": " in ret_line:                                     # Azure also changed something in the label
            new_value = ret_line.split(": ", 1)[1].strip()
        else:
            new_value = ret_line.strip()
        it["deid_value"] = new_value
        it["changed"] = new_value != it["value"]
    return True

print("✓ Azure DeID functions defined (Surrogate)")

## 8. Process One Image

`process_image(image_path)` runs steps 1–7 for one image and writes all files to the image's folder (or to `OUTPUT_SUBFOLDER` inside it). If the GPT call fails, the OCR text still goes to Azure and the error is saved.

In [ ]:
SUPPORTED_IMAGE_FORMATS = ['.png', '.jpg', '.jpeg', '.bmp', '.gif', '.tif', '.tiff']
OUTPUT_SUBFOLDER = ""        # "" = save next to the image; e.g. "deid_step1" = a sub-folder of the image's folder


def _write(path, text):
    path.write_text(text, encoding="utf-8")
    return path

def _write_json(path, data):
    path.write_text(json.dumps(data, indent=1, ensure_ascii=False, default=str), encoding="utf-8")
    return path


def process_image(image_path):
    image_path = Path(image_path)
    if not image_path.exists():
        raise FileNotFoundError(f"Image file not found: {image_path}")
    if image_path.suffix.lower() not in SUPPORTED_IMAGE_FORMATS:
        raise ValueError(f"Unsupported format: {image_path.suffix}")
    out_dir = image_path.parent / OUTPUT_SUBFOLDER if OUTPUT_SUBFOLDER else image_path.parent
    out_dir.mkdir(parents=True, exist_ok=True)
    stem = image_path.stem
    info = {"image_path": str(image_path), "output_dir": str(out_dir), "errors": {}, "seconds": {}}
    AZURE_USAGE.update(calls=0, input_bytes=0)

    # 1-2. Textract OCR + OCR text
    t0 = time.time()
    image_bytes = image_path.read_bytes()
    if len(image_bytes) >= 10 * 1024 * 1024:
        raise ValueError(f"Image exceeds Textract's 10 MB limit: {image_path.name}")
    blocks = get_ocr(image_bytes)["Blocks"]
    with Image.open(image_path) as im:
        img_w, img_h = im.size
    full_text, offset_map = build_word_index(blocks, img_w, img_h)
    info["seconds"]["ocr"] = round(time.time() - t0, 2)
    _write(out_dir / f"{stem}_ocr_text.txt", full_text)
    _write_json(out_dir / f"{stem}_ocr_words.json", {
        "image_size": [img_w, img_h],
        "words": [{"word_id": w["word_id"], "idx": w["idx"], "text": w["text"], "line_idx": w["line_idx"],
                   "start": w["start"], "end": w["end"], "px": list(w["px"]), "angle": round(w["angle"], 2),
                   "confidence": round(w["confidence"], 2), "handwriting": w["handwriting"]} for w in offset_map]})

    # 3-4. GPT-5.4 extraction + GPT text
    t0 = time.time()
    gpt, gpt_text, items = {}, "", []
    if USE_VLM:
        try:
            gpt = vlm_extract(image_path)
            gpt_text, items = build_gpt_text(gpt)
        except Exception as exc:
            info["errors"]["gpt"] = repr(exc)[:300]
            print(f"⚠ GPT step failed for {image_path.name}: {exc!r}")
    info["seconds"]["gpt"] = round(time.time() - t0, 2)
    _write_json(out_dir / f"{stem}_gpt.json", {"model": VLM_MODEL_NAME, "deployment": AZURE_OPENAI_DEPLOYMENT,
                                              "error": info["errors"].get("gpt", ""), "answer": gpt})
    _write(out_dir / f"{stem}_gpt_text.txt", gpt_text)

    # 5. Azure DeID Surrogate on both texts (two separate calls)
    t0 = time.time()
    deid_ocr = deid_surrogate(full_text)
    _write(out_dir / f"{stem}_deid_ocr_text.txt", deid_ocr)
    deid_gpt = ""
    if gpt_text.strip():
        deid_gpt = deid_surrogate(gpt_text)
    _write(out_dir / f"{stem}_deid_gpt_text.txt", deid_gpt)
    info["seconds"]["deid"] = round(time.time() - t0, 2)

    # 6. GPT item -> Azure's value
    paired = pair_gpt_items(items, gpt_text, deid_gpt) if items else True
    _write_json(out_dir / f"{stem}_gpt_deid_items.json", items)

    # 7. run info
    ocr_changed_lines = sum(a != b for a, b in zip(full_text.split("\n"), deid_ocr.split("\n")))
    info.update({
        "ocr_words": len(offset_map), "ocr_lines": len(full_text.split("\n")) - 1,
        "ocr_lines_changed_by_azure": ocr_changed_lines,
        "gpt_fields": len(gpt.get("fields", []) or []), "gpt_free_text_phi": len(gpt.get("free_text_phi", []) or []),
        "gpt_items_sent": len(items), "gpt_items_changed_by_azure": sum(1 for it in items if it.get("changed")),
        "gpt_phi_items_not_changed": sum(1 for it in items if it.get("is_phi") and it.get("changed") is False),
        "gpt_pairing_ok": paired,
        "deid_calls": AZURE_USAGE["calls"], "deid_input_bytes": AZURE_USAGE["input_bytes"],
    })
    _write_json(out_dir / f"{stem}_run_info.json", info)
    return info


def print_report(info):
    print(f"Image: {info['image_path']}")
    print(f"Saved to: {info['output_dir']}")
    for k in ("ocr_words", "ocr_lines", "ocr_lines_changed_by_azure", "gpt_fields", "gpt_free_text_phi",
              "gpt_items_sent", "gpt_items_changed_by_azure", "gpt_phi_items_not_changed", "gpt_pairing_ok",
              "deid_calls", "deid_input_bytes", "seconds", "errors"):
        print(f"  {k:28}: {info.get(k)}")

print("✓ process_image defined")

## 9. Run on One Image

Set `IMAGE_PATH` and run. The report lists what was found and changed, and the files are in the image's folder. `SHOW_TEXTS = True` also prints the four texts (OCR text, Azure's output for it, GPT text, Azure's output for it) — **they contain real PHI**.

In [ ]:
IMAGE_PATH = r"C:\Users\RSahu\Documents\Azure_deidentification\Input_docs\data\test\CMS 1500 202604230008171_page_1.png"
SHOW_TEXTS = True

info = process_image(IMAGE_PATH)
print_report(info)

if SHOW_TEXTS:
    _dir, _stem = Path(info["output_dir"]), Path(IMAGE_PATH).stem
    for suffix, title in (("_ocr_text.txt", "OCR text sent to Azure"), ("_deid_ocr_text.txt", "Azure output for the OCR text"),
                          ("_gpt_text.txt", "GPT text sent to Azure"), ("_deid_gpt_text.txt", "Azure output for the GPT text")):
        print("\n" + "=" * 100 + f"\n{title}\n" + "=" * 100)
        print((_dir / f"{_stem}{suffix}").read_text(encoding="utf-8"))

## 10. Run on a Folder (optional)

Processes every image in `BATCH_INPUT_DIR` (and sub-folders); each image's files go next to that image. A page that fails is logged and the loop continues. `step1_summary.csv` (one row per image) is written in `BATCH_INPUT_DIR`.

In [ ]:
import csv

BATCH_INPUT_DIR = Path(r"C:\Users\RSahu\Documents\Azure_deidentification\Input_docs\data")
BATCH_RECURSIVE = True

images = sorted(p for p in BATCH_INPUT_DIR.glob("**/*" if BATCH_RECURSIVE else "*")
                if p.is_file() and p.suffix.lower() in SUPPORTED_IMAGE_FORMATS
                and not p.stem.lower().endswith(("_deid", "_comparison")) and p.parent.name.lower() != "output_images")
print(f"Found {len(images)} image(s)")

rows, failed = [], []
for n, image_path in enumerate(images, 1):
    print(f"[{n}/{len(images)}] {image_path.name}")
    try:
        info = process_image(image_path)
        rows.append({k: (json.dumps(v) if isinstance(v, dict) else v) for k, v in info.items()})
    except Exception as exc:
        failed.append({"image_path": str(image_path), "error": repr(exc)})
        print(f"    FAILED: {exc!r}")

if rows:
    with open(BATCH_INPUT_DIR / "step1_summary.csv", "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=list(rows[0].keys()))
        writer.writeheader(); writer.writerows(rows)
print(f"Done: {len(rows)} ok, {len(failed)} failed | summary: {BATCH_INPUT_DIR / 'step1_summary.csv'}")
for f in failed:
    print("  ", f)